In [1]:
%%bash
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH=$HOME/.local/bin:$PATH
rm -rf /tmp/repo
git clone -q -b ajak-evaluation-docs https://github.com/Eddydev-ALU/freeway-dqn-formative-3.git /tmp/repo
cd /tmp/repo
uv sync -q
uv run python -c "import torch; assert torch.cuda.is_available(), 'NO GPU - do not commit'; print('GPU OK:', torch.cuda.device_count(), torch.cuda.get_device_name(0))"

installing to /usr/local/bin
  uv
  uvx
everything's installed!
GPU OK: 2 Tesla T4


downloading uv 0.13.8 x86_64-unknown-linux-gnu


In [2]:
%%bash
export PATH=$HOME/.local/bin:$PATH
cd /tmp/repo
OUT=/kaggle/working/Ajak_Chol
mkdir -p $OUT

# run <gpu> <exp-name> <hyperparameters...>; console output goes to <exp>_stdout.txt
run() {
  gpu=$1; exp=$2; shift 2
  echo "$(date +%H:%M) start $exp on GPU $gpu"
  CUDA_VISIBLE_DEVICES=$gpu uv run python train.py "$@" --output-dir $OUT --exp-name $exp > $OUT/${exp}_stdout.txt 2>&1
  echo "$(date +%H:%M) done  $exp: $(tail -1 $OUT/${exp}_log.csv)"
}

# 3 workers run in parallel, each working through its own list
(
run 0 exp01 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 1.0 --eps-end 0.10 --eps-decay 250000
run 0 exp04 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 1.0 --eps-end 0.01 --eps-decay 250000
run 0 exp07 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 0.5 --eps-end 0.10 --eps-decay 250000
run 0 exp08 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 0.1 --eps-end 0.01 --eps-decay 50000
) &
(
run 1 exp02 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 1.0 --eps-end 0.20 --eps-decay 250000
run 1 exp05 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 1.0 --eps-end 0.10 --eps-decay 100000
run 1 exp10 --lr 1e-4 --gamma 0.97 --batch-size 128 --eps-start 0.8 --eps-end 0.10 --eps-decay 250000
) &
(
run 0 exp03 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 1.0 --eps-end 0.30 --eps-decay 250000
run 1 exp06 --lr 1e-4 --gamma 0.99 --batch-size 64  --eps-start 1.0 --eps-end 0.10 --eps-decay 400000
run 0 exp09 --lr 5e-5 --gamma 0.99 --batch-size 32  --eps-start 1.0 --eps-end 0.20 --eps-decay 250000
) &
wait

# Greedy evaluation of every model, the same way play.py runs in the demo
for i in 01 02 03 04 05 06 07 08 09 10; do
  echo "exp$i: $(uv run python play.py --model $OUT/exp${i}_model.pth --no-render --episodes 3 2>/dev/null | tail -1)"
done | tee $OUT/greedy_eval.txt

01:53 start exp02 on GPU 1
01:53 start exp03 on GPU 0
01:53 start exp01 on GPU 0
02:22 done  exp02: 244,499712,18.0,2048,0.2
02:22 start exp05 on GPU 1
02:25 done  exp03: 244,499712,17.0,2048,0.3
02:25 start exp06 on GPU 1
02:25 done  exp01: 244,499712,20.0,2048,0.1
02:25 start exp04 on GPU 0
02:55 done  exp04: 244,499712,25.0,2048,0.01
02:55 start exp07 on GPU 0
02:55 done  exp05: 244,499712,21.0,2048,0.1
02:55 start exp10 on GPU 1
02:56 done  exp06: 244,499712,19.0,2048,0.1
02:56 start exp09 on GPU 0
03:27 done  exp09: 244,499712,20.0,2048,0.2
03:28 done  exp07: 244,499712,25.0,2048,0.1
03:28 start exp08 on GPU 0
03:29 done  exp10: 244,499712,31.0,2048,0.1
03:52 done  exp08: 244,499712,29.0,2048,0.01
exp01: Average score over 3 game(s): 21.0
exp02: Average score over 3 game(s): 21.0
exp03: Average score over 3 game(s): 18.7
exp04: Average score over 3 game(s): 21.0
exp05: Average score over 3 game(s): 21.0
exp06: Average score over 3 game(s): 21.0
exp07: Average score over 3 game(s):